# Módulo 02 · Aula 3 — Filtros e agrupamentos

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Você já sabe abrir uma tabela e olhar para ela. Agora vamos **fazer perguntas** a ela.

Quase toda pergunta sobre dados tem a mesma forma: *recorte um pedaço* (filtro) e
*resuma esse pedaço* (agregação). "Qual foi o retorno médio das ações do setor
financeiro em 2024?" é exatamente isso.

Ao final desta aula você vai saber:

- filtrar linhas por uma ou várias condições;
- usar `isin`, `between`, `.str.contains` e `query`;
- extrair ano, mês e dia de uma coluna de data com `.dt`;
- agrupar com **`groupby`** e agregar com uma ou várias funções;
- montar tabelas cruzadas com `pivot_table`.

**Tempo estimado:** 75 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "02_Manipulacao_Dados"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
clientes = pd.read_csv("../data/clientes_corretora.csv")

print("acoes   :", acoes.shape)
print("clientes:", clientes.shape)
acoes.head(3)

## 1. Filtrando: a máscara booleana

O mecanismo é o mesmo do NumPy. Uma comparação sobre a coluna produz uma Series de
`True`/`False`; usar essa Series entre colchetes devolve apenas as linhas `True`.

In [ ]:
mascara = acoes["ticker"] == "PETR4"

print(mascara.head())
print()
print("Linhas selecionadas:", mascara.sum(), "de", len(acoes))

In [ ]:
petr = acoes[acoes["ticker"] == "PETR4"]
petr.head(3)

### Várias condições

Use `&` (e), `|` (ou), `~` (não). **Cada condição entre parênteses** — sem eles, o
Python aplica a precedência errada e o resultado é um erro confuso.

In [ ]:
# Ações da PETR4 com volume acima de 80 milhões
filtro = acoes[(acoes["ticker"] == "PETR4") & (acoes["volume"] > 80_000_000)]
print(f"{len(filtro)} pregões atendem às duas condições")
filtro.head(3)

In [ ]:
# Duas empresas quaisquer, no mesmo filtro
bancos = acoes[(acoes["ticker"] == "ITUB4") | (acoes["ticker"] == "BBDC4")]
print(bancos["ticker"].value_counts())

In [ ]:
# isin() é muito mais limpo quando são vários valores
bancos = acoes[acoes["ticker"].isin(["ITUB4", "BBDC4", "B3SA3"])]
print(bancos["ticker"].unique())

# between() para faixas — inclui as duas pontas
faixa = acoes[acoes["fechamento"].between(30, 35)]
print(f"{len(faixa)} pregões com fechamento entre 30 e 35")

In [ ]:
# ~ nega uma condição inteira
sem_bancos = acoes[~acoes["ticker"].isin(["ITUB4", "BBDC4", "B3SA3"])]
print(sorted(sem_bancos["ticker"].unique()))

### Filtrando texto

O acessador `.str` dá acesso aos métodos de string coluna afora.

In [ ]:
print(clientes["cidade"].dropna().unique()[:10])

Repare no que apareceu: a mesma cidade escrita de várias formas. Isso muda o resultado
de qualquer filtro de texto.

In [ ]:
# .str.contains: a coluna contém este trecho?
paulistas = clientes[clientes["cidade"].str.contains("Paulo", na=False)]
print(f"Com .contains('Paulo')            : {len(paulistas)} clientes")

# case=False ignora maiúsculas e minúsculas
paulistas = clientes[clientes["cidade"].str.contains("paulo", case=False, na=False)]
print(f"Com .contains('paulo', case=False): {len(paulistas)} clientes")

A diferença entre os dois números é o tamanho do erro que você cometeria sem perceber.

Dois parâmetros salvam a vida aqui:

- **`case=False`** ignora a diferença entre maiúsculas e minúsculas;
- **`na=False`** define o que fazer nas linhas em que a cidade está faltando. Sem ele,
  o filtro quebra ao encontrar `NaN`.

E note que nem isso resolve tudo: `"São Paulo"` e `"Sao Paulo"` continuam diferentes
para o `.contains("São")`. A solução de verdade é **padronizar a coluna antes de
filtrar** — assunto da próxima aula.

In [ ]:
# .str.startswith / .str.endswith
print(clientes[clientes["nome"].str.startswith("A")].head(3)[["nome", "cidade"]])

### `query`: a sintaxe alternativa

`query` permite escrever a condição como texto. Fica mais legível quando há várias
condições, e evita a floresta de parênteses.

In [ ]:
resultado_1 = acoes[(acoes["ticker"] == "VALE3") & (acoes["fechamento"] > 70)]
resultado_2 = acoes.query("ticker == 'VALE3' and fechamento > 70")

print(len(resultado_1), len(resultado_2))
resultado_2.head(3)

Os dois estilos são corretos e você vai encontrar ambos. Use o que ficar mais legível no
caso concreto.

## 2. Trabalhando com datas: o acessador `.dt`

Como a coluna `data` foi lida com `parse_dates`, ela é do tipo `datetime` e dá acesso a
`.dt` — o equivalente do `.str`, mas para datas.

In [ ]:
print(acoes["data"].dtype)

acoes["ano"] = acoes["data"].dt.year
acoes["mes"] = acoes["data"].dt.month
acoes["dia_semana"] = acoes["data"].dt.day_name()
acoes["ano_mes"] = acoes["data"].dt.to_period("M")   # útil para agrupar por mês

acoes[["data", "ano", "mes", "dia_semana", "ano_mes"]].head()

In [ ]:
# Filtrando por período
pregoes_2025 = acoes[acoes["ano"] == 2025]
print(f"{len(pregoes_2025)} registros em 2025")

primeiro_trimestre = acoes[(acoes["data"] >= "2025-01-01") & (acoes["data"] <= "2025-03-31")]
print(f"{len(primeiro_trimestre)} registros no 1º trimestre de 2025")

> **Dica:** Repare que dá para comparar uma coluna `datetime` diretamente com um texto no
> formato `"2025-01-01"`. O pandas converte sozinho.

## 3. `groupby`: a operação mais importante do pandas

`groupby` implementa um padrão chamado **dividir–aplicar–combinar**:

1. **dividir** a tabela em grupos, segundo uma coluna;
2. **aplicar** uma função a cada grupo;
3. **combinar** os resultados em uma tabela nova.

Você já fez isso na mão, na aula 01.3, com um dicionário e um `for`. Agora é uma linha.

In [ ]:
# Preço médio de fechamento por ativo
acoes.groupby("ticker")["fechamento"].mean().round(2)

Leia a expressão em três partes:

```python
acoes.groupby("ticker")   ["fechamento"]   .mean()
#     └── divide por      └── nesta        └── aplica
#         ticker              coluna           esta função
```

O resultado é uma Series cujo **índice** são os grupos.

In [ ]:
# Várias estatísticas de uma vez, com .agg()
acoes.groupby("ticker")["fechamento"].agg(["count", "mean", "std", "min", "max"]).round(2)

In [ ]:
# Agregando colunas diferentes com funções diferentes
acoes.groupby("ticker").agg({
    "fechamento": "mean",
    "volume": "sum",
    "data": "max",
}).round(2)

In [ ]:
# Agregação nomeada: o jeito mais legível, e o que dá nomes decentes às colunas
resumo = acoes.groupby("ticker").agg(
    pregoes=("data", "count"),
    preco_medio=("fechamento", "mean"),
    preco_minimo=("fechamento", "min"),
    preco_maximo=("fechamento", "max"),
    volume_medio=("volume", "mean"),
).round(2)

resumo.sort_values("preco_medio", ascending=False)

### Agrupando por mais de uma coluna

In [ ]:
# Preço médio por ativo e por ano
medias = acoes.groupby(["ticker", "ano"])["fechamento"].mean().round(2)
medias.head(10)

O resultado tem um índice de dois níveis (*MultiIndex*). Para voltar ao formato de tabela
comum, use `reset_index()`:

In [ ]:
medias_tabela = medias.reset_index()
medias_tabela.head()

In [ ]:
# Ou peça o resultado já assim, com as_index=False
acoes.groupby(["ticker", "ano"], as_index=False)["fechamento"].mean().round(2).head()

### Um caso completo: retorno anual por ativo

Aqui `groupby` responde a uma pergunta financeira de verdade. Para cada ativo e cada
ano, queremos o retorno do período — ou seja, quanto variou entre o primeiro e o último
pregão.

In [ ]:
def retorno_do_periodo(serie_de_precos):
    """Variação percentual entre o primeiro e o último preço da série."""
    return (serie_de_precos.iloc[-1] - serie_de_precos.iloc[0]) / serie_de_precos.iloc[0]


# Garantimos a ordem cronológica antes de agrupar — o "primeiro" e o "último"
# dependem disso.
acoes_ordenadas = acoes.sort_values("data")

retornos_anuais = (
    acoes_ordenadas
    .groupby(["ticker", "ano"])["fechamento_ajustado"]
    .apply(retorno_do_periodo)
    .reset_index(name="retorno")
)

retornos_anuais["retorno"] = (retornos_anuais["retorno"] * 100).round(1)
retornos_anuais.head(10)

> Usamos `fechamento_ajustado` e não `fechamento`. O preço ajustado incorpora dividendos
> e desdobramentos, então é ele que mede o retorno de quem investiu. Usar o preço bruto
> subestima o retorno de ações que pagam muito dividendo — como as do nosso conjunto.
> Detalhe pequeno na sintaxe, grande na conclusão.

## 4. `pivot_table`: a tabela dinâmica

Se você já usou tabela dinâmica no Excel, é exatamente isso: uma variável nas linhas,
outra nas colunas, um valor agregado no meio.

In [ ]:
tabela = retornos_anuais.pivot_table(
    index="ticker",       # o que vai nas linhas
    columns="ano",        # o que vai nas colunas
    values="retorno",     # o que preencher
    aggfunc="mean",       # como agregar (aqui há um valor por célula, mas o parâmetro é obrigatório)
)
tabela

Uma tabela dessas responde muita coisa de relance: 2021 e 2022 foram bons para
commodities e ruins para varejo; MGLU3 despencou em anos seguidos; WEGE3 e os bancos
foram mais estáveis. Guardaremos essas observações para o módulo 03, quando elas viram
gráficos e hipóteses.

In [ ]:
# margins=True acrescenta totais/médias nas bordas
retornos_anuais.pivot_table(
    index="ticker", columns="ano", values="retorno",
    aggfunc="mean", margins=True, margins_name="MÉDIA",
).round(1)

## 5. Praticando com a base de clientes

Uma base diferente, com perguntas de negócio em vez de mercado. Repare que esta base é
**suja** — os perfis aparecem escritos de várias formas. Vamos conviver com isso agora e
resolver na próxima aula.

In [ ]:
clientes.head(3)

In [ ]:
print(clientes["perfil_investidor"].value_counts(dropna=False))

Aí está o problema: `Conservador`, `CONSERVADOR`, `conservador` e `  Conservador ` são
quatro categorias distintas para o pandas. Qualquer agrupamento sairia errado.

Uma padronização mínima resolve por enquanto:

In [ ]:
clientes["perfil"] = clientes["perfil_investidor"].str.strip().str.title()
clientes["perfil"].value_counts(dropna=False)

In [ ]:
# Distribuição relativa
clientes["perfil"].value_counts(normalize=True).mul(100).round(1)

In [ ]:
# Clientes por estado e perfil
clientes.groupby(["estado", "perfil"]).size().unstack(fill_value=0)

> `.size()` conta linhas por grupo. `.unstack()` transforma o último nível do índice em
> colunas — é uma forma alternativa de chegar ao mesmo lugar que `pivot_table`.

In [ ]:
# Perguntas de negócio, uma a uma
print("--- Idade média por perfil ---")
print(clientes.groupby("perfil")["idade"].mean().round(1))

print("\n--- Aporte mensal médio por estado (top 5) ---")
print(clientes.groupby("estado")["aporte_mensal"].mean().round(2).nlargest(5))

print("\n--- Clientes por situação ---")
print(clientes["ativo"].value_counts())

Repare no último resultado: a coluna `ativo` tem `sim`, `nao`, `1` e `0` — quatro
valores para duas situações. E a `idade` média inclui as idades impossíveis que vimos no
`describe`. **Nenhum agrupamento é melhor que os dados que entram nele.** É isso que a
próxima aula resolve.

## 6. Recapitulando

- Filtrar é aplicar uma **máscara booleana**: `df[df["col"] > valor]`.
- Várias condições: `&`, `|`, `~`, cada uma entre parênteses. Para muitos valores, use
  `isin`; para faixas, `between`; para texto, `.str.contains(..., na=False)`.
- `query("col > 10 and outra == 'X'")` é uma alternativa mais legível.
- `.dt` extrai ano, mês, dia da semana e período de colunas de data.
- **`groupby`** divide–aplica–combina. Use `.agg()` com agregação nomeada para
  resultados legíveis, e `reset_index()` para voltar ao formato de tabela.
- `pivot_table` monta tabela dinâmica: `index`, `columns`, `values`, `aggfunc`.
- Agrupamento sobre dados inconsistentes produz resultado inconsistente.

**Próxima aula:** juntar tabelas com `merge` e limpar dados de verdade.